# Bloque residual sobre CIFAR-10

Ejemplo compacto con la API funcional de Keras.

## Dependencias

En un entorno nuevo: `%pip install tensorflow numpy matplotlib`. La primera carga descarga CIFAR-10.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 2026
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train[:20_000].astype('float32') / 255.0
y_train = y_train[:20_000].squeeze()
x_test = x_test.astype('float32') / 255.0
y_test = y_test.squeeze()

## Bloque residual

La proyección $1\times1$ se usa cuando cambia el número de canales o el stride.

In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding='same', use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    y = layers.Conv2D(filters, 3, padding='same', use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if stride != 1 or x.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    return layers.ReLU()(layers.Add()([y, shortcut]))

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = layers.Conv2D(32, 3, padding='same', use_bias=False)(inputs)
x = layers.BatchNormalization()(x)
x = layers.ReLU()(x)
x = residual_block(x, 32)
x = residual_block(x, 64, stride=2)
x = residual_block(x, 64)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(10)(x)
model = keras.Model(inputs, outputs, name='mini_resnet')
model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)
model.summary()

In [ ]:
history = model.fit(
    x_train, y_train, validation_split=0.15,
    epochs=8, batch_size=128, verbose=2,
    callbacks=[keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
)

In [ ]:
plt.plot(history.history['accuracy'], label='train')
plt.plot(history.history['val_accuracy'], label='validación')
plt.xlabel('Época'); plt.ylabel('Accuracy'); plt.legend(); plt.grid(alpha=0.3)
plt.show()
print('Prueba:', model.evaluate(x_test, y_test, verbose=0))

## Trabajo propuesto

Construya una CNN sin atajos con un número comparable de parámetros. Compare varias semillas, curva respecto de tiempo y accuracy por clase.